In [1]:
import re, csv, os, glob
import pandas as pd

# ── FUNCIÓN LECTORA (misma que el script anterior) ───────────────────────────
def leer_csv_capology(archivo):
    filas = []
    with open(archivo, 'r', encoding='utf-8-sig', errors='ignore', newline='') as f:
        for linea in f:
            linea = linea.rstrip('\r\n').strip()
            if not linea:
                continue
            if linea.startswith('"') and linea.endswith('"'):
                linea = linea[1:-1]
            linea = re.sub(r'""([^"]*?)""', r'"\1"', linea)
            parsed = next(csv.reader([linea]))
            filas.append(parsed)
    return filas


# ── LOOP 2025-2026 ───────────────────────────────────────────────────────────
# Ruta separada porque estos archivos están en una carpeta distinta
ruta_2526 = r'C:\Users\PAOLA\Desktop\Proyectos personales\Rendimiento_vs_salario\Salarios\salarios_2025_2026'
ligas = ['La_liga', 'Premier', 'Bundesliga', 'Serie_a', 'Ligue']
acumulado = []

for archivo in glob.glob(os.path.join(ruta_2526, "*.csv")):
    # Detectar liga desde el nombre del archivo
    nombre_arc = os.path.basename(archivo).lower()
    liga = next((l for l in ligas if l.lower() in nombre_arc), 'Desconocida')

    try:
        filas = leer_csv_capology(archivo)
        df = pd.DataFrame(filas[1:], columns=filas[0])

        # 19 cols: [0]Player [1]skip [2]Weekly [3]Annual [4]Bonus [5]Total
        #          [6]Status [7]Signed [8]Expiration [9]YearsRemaining
        #          [10]NetRemaining [11]ReleaseClause [12]Pos [13]PosDetalle
        #          [14]Age [15]Country [16]Club [17]Active [18]Loan
        indices = [0, 2, 3, 4, 5, 12, 14, 15, 16]
        df = df.iloc[:, indices].copy()
        df.columns = [
            'Player', 'Net_Fixed_PW_USD', 'Net_Fixed_PY_USD',
            'Net_Bonus_PY_USD', 'Net_Total_PY_USD',
            'Position', 'Age', 'Country', 'Club'
        ]

        cols_dinero = ['Net_Fixed_PW_USD', 'Net_Fixed_PY_USD',
                       'Net_Bonus_PY_USD', 'Net_Total_PY_USD']
        for col in cols_dinero:
            df[col] = df[col].str.replace(r'[^\d.]', '', regex=True)
            df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)

        df['Age'] = pd.to_numeric(df['Age'], errors='coerce')
        df['Temporada'] = '2025-2026'
        df['Liga'] = liga

        df = df[df['Player'].str.strip() != '']
        df = df[df['Net_Total_PY_USD'] > 0]

        acumulado.append(df)
        print(f"✅ {os.path.basename(archivo)}: {len(df)} jugadores ({liga})")

    except Exception as e:
        print(f"⚠️ Fallo en {os.path.basename(archivo)}: {e}")


# ── UNIÓN Y EXPORTACIÓN ──────────────────────────────────────────────────────
if acumulado:
    df_master_2526 = pd.concat(acumulado, ignore_index=True)

    cols_orden = ['Player', 'Position', 'Age', 'Country', 'Club',
                  'Net_Fixed_PW_USD', 'Net_Fixed_PY_USD', 'Net_Bonus_PY_USD',
                  'Net_Total_PY_USD', 'Temporada', 'Liga']
    df_master_2526 = df_master_2526[cols_orden]

    ruta_final = r'C:\Users\PAOLA\Desktop\Proyectos personales\Rendimiento_vs_salario\Dashboard_Data\Master_Salarios_2526.csv'
    df_master_2526.to_csv(ruta_final, index=False, encoding='utf-8-sig')

    print(f"\n✅ Guardado: {len(df_master_2526)} registros")
    print(f"\nPor liga:\n{df_master_2526['Liga'].value_counts().to_string()}")
    display(df_master_2526.head())
else:
    print("🛑 No se procesó ningún archivo.")

✅ capology_bundesliga_25_26_raw.csv: 545 jugadores (Bundesliga)
✅ capology_laliga_25_26_raw.csv: 522 jugadores (Desconocida)
✅ capology_ligue_25_26_raw.csv: 497 jugadores (Ligue)
✅ capology_premier_25_26_raw.csv: 681 jugadores (Premier)
✅ capology_seriea_25_26_raw.csv: 582 jugadores (Desconocida)

✅ Guardado: 2827 registros

Por liga:
Liga
Desconocida    1104
Premier         681
Bundesliga      545
Ligue           497


,Player,Position,Age,Country,Club,Net_Fixed_PW_USD,Net_Fixed_PY_USD,Net_Bonus_PY_USD,Net_Total_PY_USD,Temporada,Liga
0,Harry Kane,F,32,England,Bayern Munich,297751.0,15483072.0,3870768.0,19353840.0,2025-2026,Bundesliga
1,Manuel Neuer,K,39,Germany,Bayern Munich,250111.0,13005780.0,3251445.0,16257226.0,2025-2026,Bundesliga
2,Jamal Musiala,F,22,Germany,Bayern Munich,247190.0,12853871.0,7011202.0,19865073.0,2025-2026,Bundesliga
3,Joshua Kimmich,M,30,Germany,Bayern Munich,238201.0,12386458.0,3096614.0,15483072.0,2025-2026,Bundesliga
4,Alphonso Davies,D,25,Canada,Bayern Munich,178651.0,9289843.0,3096614.0,12386458.0,2025-2026,Bundesliga
